# Importation des données

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
data_path = '/content/drive/MyDrive/Marathon/data/'

# Librairies

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df_conso = pd.read_parquet(data_path+'choix_vehicule.parquet')
df_color = pd.read_parquet(data_path+'auto_color.parquet')

# Exploration des données

In [5]:
color = df_color.drop(columns=['Catégorie', 'Code de couleur', 'Element de carrosserie','Couleur apprêt', 'Variante','Système'])
conso = df_conso.drop(columns=['usage', 'presentation_video', 'reprise_energie', 'reprise_annee', 'reprise_modele', 'kms_min','kms_max', 'kms_reprise', 'modele_concurrence', 'marque_concurrence'])


In [6]:
conso.info()

<class 'pandas.core.frame.DataFrame'>
Index: 211468 entries, 0 to 123113
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   annee             211468 non-null  int64 
 1   mois              211468 non-null  int64 
 2   civilite          211468 non-null  object
 3   departement       211468 non-null  object
 4   brand             211468 non-null  object
 5   modele            211468 non-null  object
 6   energie           211468 non-null  object
 7   boite_de_vitesse  211468 non-null  object
 8   reprise_marque    211468 non-null  object
 9   reprise_couleur   211468 non-null  object
dtypes: int64(2), object(8)
memory usage: 17.7+ MB


In [7]:
for col in conso.columns :
  dips=conso[col].value_counts()
  display(dips)

,count
annee,
2025,123114
2024,80709
2023,7645


,count
mois,
11,31261
10,25719
9,21890
6,20436
3,18344
4,16912
7,15534
5,15132
1,14065


,count
civilite,
monsieur,149875
madame,61302
societe,290
NC,1


,count
departement,
13,11319
93,9246
59,8583
69,7535
94,6877
...,...
Ma,1
0B,1
0n,1


,count
brand,
Volkswagen,53006
Citroen,40526
AUDI,20489
CUPRA,18909
Hyundai,18510
Skoda,11510
Seat,7458
28121657398034,4186
Fiat,4033


,count
modele,
Ami,14041
C3a,12452
Cupra formentor,9330
T-roc 2025,8677
T-roc,6141
...,...
Id.5 gtx 2025,1
Tayron hev,1
Kona ev,1


,count
energie,
essence,142869
hybride,21933
diesel,19926
électrique,16957
non_déterminé,9783


,count
boite_de_vitesse,
robotisée,175977
mecanique,35491


,count
reprise_marque,
reprise_non,127339
Peugeot,13667
Renault,12302
Citroën,11734
Volkswagen,7756
Audi,4341
Mercedes benz,3515
Opel,3051
pas de reprise,3031


,count
reprise_couleur,
NC,205828
gris,2021
noir,1120
blanc,1002
bleu,551
rouge,346
marron,189
vert,176
beige,136


In [8]:
"""ids_a_verifier = [
    '25091010376082', '25382330714770', '26835333024274',
    '26881567913874', '27006279306642', '28121657398034',
    '29481365320082', '29576441776146', '29577564225426',
    '29577629036690'
]

print("--- Inspection des modèles par ID ---")
for id_auto in ids_a_verifier:
    if id_auto in marques.index:
        print(f"ID {id_auto} contient : {marques[id_auto]}")
    else:
        print(f"ID {id_auto} non trouvé dans le dataset.")
"""

'ids_a_verifier = [\n    \'25091010376082\', \'25382330714770\', \'26835333024274\',\n    \'26881567913874\', \'27006279306642\', \'28121657398034\',\n    \'29481365320082\', \'29576441776146\', \'29577564225426\',\n    \'29577629036690\'\n]\n\nprint("--- Inspection des modèles par ID ---")\nfor id_auto in ids_a_verifier:\n    if id_auto in marques.index:\n        print(f"ID {id_auto} contient : {marques[id_auto]}")\n    else:\n        print(f"ID {id_auto} non trouvé dans le dataset.")\n'

In [9]:
mask_stellantis = (conso['brand'] == '27006279306642')

conso['brand'] = np.where(
    mask_stellantis & conso['modele'].str.contains('^2|^3|^4|^5', regex=True),
    'Peugeot',
    np.where(
        mask_stellantis & conso['modele'].str.contains('^C3', case=False, regex=True),
        'Citroen',
        conso['brand']
    )
)
masque_id = (conso['brand'] == '26835333024274')

conso['brand'] = np.where(
    masque_id & conso['modele'].str.contains('ypsilon', case=False),
    'Lancia',
    np.where(
        masque_id & conso['modele'].str.contains('audi|q2', case=False),
        'Audi',
        conso['brand']
    )
)

mapping_marques = {
    'Audi AGC': 'AUDI',
    'LeadGen Mitsubishi': 'Mitsubishi',
    'Lexus WCB': 'LEXUS',
    'Nissan Lead Gen': 'Nissan',
    'Renault Annalect': 'Renault',
    'Vidéo Live Volkswagen': 'Volkswagen',
    'Mercedes-benz': 'Mercedes',
    'Mercedes-Benz': 'Mercedes',
    'VW Utilitaires':'VWU',
    'Alpine France': 'Alpine',
    '25091010376082':'Honda',
    '25382330714770':'AUDI',
    '26881567913874':'Hyundai',
    '28121657398034':'Skoda',
     '29481365320082':'Volkswagen',
    '29576441776146':'VWU',
    '29577564225426':'Seat',
    '29577629036690':'CUPRA',
    'Audi':'AUDI',
    'Mercedes-Benz ': 'Mercedes'
}

conso['brand']=conso['brand'].replace(mapping_marques)

conso = conso[conso['brand']!= 'Test E Car']



In [10]:
conso.info()

<class 'pandas.core.frame.DataFrame'>
Index: 211462 entries, 0 to 123113
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   annee             211462 non-null  int64 
 1   mois              211462 non-null  int64 
 2   civilite          211462 non-null  object
 3   departement       211462 non-null  object
 4   brand             211462 non-null  object
 5   modele            211462 non-null  object
 6   energie           211462 non-null  object
 7   boite_de_vitesse  211462 non-null  object
 8   reprise_marque    211462 non-null  object
 9   reprise_couleur   211462 non-null  object
dtypes: int64(2), object(8)
memory usage: 17.7+ MB


In [11]:
conso_filtre = conso[conso['reprise_couleur']!= "NC"].copy()

In [12]:
conso_filtre.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5640 entries, 2999 to 122982
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   annee             5640 non-null   int64 
 1   mois              5640 non-null   int64 
 2   civilite          5640 non-null   object
 3   departement       5640 non-null   object
 4   brand             5640 non-null   object
 5   modele            5640 non-null   object
 6   energie           5640 non-null   object
 7   boite_de_vitesse  5640 non-null   object
 8   reprise_marque    5640 non-null   object
 9   reprise_couleur   5640 non-null   object
dtypes: int64(2), object(8)
memory usage: 484.7+ KB
